<a href="https://colab.research.google.com/github/umamage/machine_learning_zoomcamp-homework/blob/main/NeuralNetworks/fashionclassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
%matplotlib inline

## Convolutional layer is used to convert an image to a vector representation
# Dense layer is used to make predictions from the vector representation of the image
# Convolutional layer applies filters to the image to determine existance of features and combination of features.
# pooling is done to reduce the size of the matrix


In [ ]:
pip install tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-manylinux_2_27_x86_64.whl.metadata (4.4 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.2-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (7.4 kB)
  Using cached grpcio-1.80.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3.8 kB)
  Using cached keras-3.14.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp312-cp312-manylinux_2_17_x86

In [2]:
from tensorflow.keras.preprocessing.image import load_img

In [3]:
# Clone the repository containing the dataset
!git clone https://github.com/alexeygrigorev/clothing-dataset-small.git /content/machine_learning_zoomcamp-homework/NeuralNetworks/clothing-dataset-small

Cloning into '/content/machine_learning_zoomcamp-homework/NeuralNetworks/clothing-dataset-small'...
remote: Enumerating objects: 3839, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (400/400), done.
remote: Total 3839 (delta 9), reused 385 (delta 0), pack-reused 3439 (from 1)
Receiving objects: 100% (3839/3839), 100.58 MiB | 41.38 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [4]:
path='/content/machine_learning_zoomcamp-homework/NeuralNetworks/clothing-dataset-small/train/shirt'
name='0a77dbed-06fb-4ba7-b1e3-24f3a41498bc.jpg'
fullname=f'{path}/{name}'
x = load_img(fullname,target_size=(299,299))

In [8]:
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.applications.xception import decode_predictions
model = Xception(weights='imagenet', include_top=True, input_shape=(299, 299, 3))

In [9]:
X = np.array([x])
X = preprocess_input(X)

In [10]:
pred = model.predict(X)
decode_predictions(pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


[[('n03877472', 'pajama', np.float32(0.25119925)),
  ('n04370456', 'sweatshirt', np.float32(0.12715083)),
  ('n02963159', 'cardigan', np.float32(0.12094468)),
  ('n03630383', 'lab_coat', np.float32(0.08603185)),
  ('n04599235', 'wool', np.float32(0.048791353))]]

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input)
train_ds = train_gen.flow_from_directory(
    '/content/machine_learning_zoomcamp-homework/NeuralNetworks/clothing-dataset-small/train',target_size=(150,150),batch_size=32)
train_ds.class_indices

Found 3068 images belonging to 10 classes.


{'dress': 0,
 'hat': 1,
 'longsleeve': 2,
 'outwear': 3,
 'pants': 4,
 'shirt': 5,
 'shoes': 6,
 'shorts': 7,
 'skirt': 8,
 't-shirt': 9}

In [13]:
X, y = next(train_ds)

In [15]:
y[:5]

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]], dtype=float32)

In [17]:
# generate validation dataset
val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input)
val_ds = val_gen.flow_from_directory(
    '/content/machine_learning_zoomcamp-homework/NeuralNetworks/clothing-dataset-small/validation',target_size=(150,150),batch_size=32, shuffle=False)

Found 341 images belonging to 10 classes.


In [20]:
base_model = Xception(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
# include_top False means do not include the dense layers since they are at the top
# whereas convolutional layers are at the bottom
base_model.trainable = False # do not train the convolutional layers
inputs = keras.Input(shape=(150, 150, 3))
base = base_model(inputs, training=False)
vectors = keras.layers.GlobalAveragePooling2D()(base)
outputs = vectors
model = keras.Model(inputs, outputs)
preds = model.predict(X)


1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


In [21]:
preds.shape

(32, 2048)

Pre-trained model is available. Trained on imagenet images.vector representation from conv. layer is useful but dense layer predicts 1000 classes. We need a dense layer that can predict 10 classes (cutomize).